# 7. Paper figures — raw-motion transformer

Component panels for Figures 1 and 2. Export SVG/PNG, then assemble in Illustrator.
Uses the held-out predictions of `block3_long_large` from notebook 06 (`evaluation_dir`).

| Paper panel | Section below |
|---|---|
| Fig 1a schematic | F1 schematic |
| Fig 1b example tracks | F1 example tracks |
| Fig 1c motion tokens | F1 motion tokens |
| Fig 1d embedding | F1 embedding |
| Fig 1e probabilities (+ movie) | F1 probabilities |
| Fig 1e extra fan | F1e extra |
| Fig 2a confusion | F2 confusion |
| Fig 2b benchmark | F2 benchmark |
| Fig 2c uncertainty | F2 uncertainty |
| Fig 2g fingerprints | F2 fingerprints |
| Fig 2h superwindow | F2 superwindow |

Turn heavy panels off with the `RUN_*` switches in the paths cell.

In [ ]:
import json
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import seaborn as sns
from IPython.display import display

sys.path.insert(0, "..")
from sptnano import config, evaluate, movies, plasticity, plots, superwindows

%matplotlib inline

paths = config.load_paths()
pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(30)
plt.style.use("seaborn-v0_8-whitegrid")
print("Imports complete")

## 0. Paths, palettes, paper switches

In [ ]:
# =============================================================================
# Held-out predictions of block3_long_large (written by notebook 06)
# =============================================================================
TRANSFORMER_BEHAVIOR_DIR = paths["evaluation_dir"]
TRANSFORMER_WINDOWED_PARQUET = TRANSFORMER_BEHAVIOR_DIR / "heldout_test_windows_with_transformer_behavior_state.parquet"
TRANSFORMER_INSTANT_PARQUET = TRANSFORMER_BEHAVIOR_DIR / "heldout_test_instant_with_transformer_behavior_state.parquet"
MODEL_VARIANT = "block3"
VARIANT = {"label": "block3_long_large — 3-frame averaged tokens", "feature_mode": "block3", "run_id": "block3_long_large"}
VARIANT_FILTER_RUN_ID = None

# Optional second-molecule tables (Catherine / kinesin / myosin). Leave None to skip.
# Example: Path(r"D:/TRANSFORMER_DEVELOPMENT/saved_data/<catherine_eval>/heldout_test_windows_with_transformer_behavior_state.parquet")
CATHERINE_WINDOWED_PARQUET = None
CATHERINE_INSTANT_PARQUET = None

PLOTS_DIR = paths["plots_dir"]
RESULTS_DIR = PLOTS_DIR / "nature_methods_transformer_2026"
FIG1_DIR = RESULTS_DIR / "figure1"
FIG2_DIR = RESULTS_DIR / "figure2"
ED_DIR = RESULTS_DIR / "extended_data"
for d in (RESULTS_DIR, FIG1_DIR, FIG2_DIR, ED_DIR):
    d.mkdir(parents=True, exist_ok=True)

FRAME_INTERVAL = config.TIME_BETWEEN_FRAMES
PIXEL_SIZE_UM = config.PIXELSIZE_MICRONS
TIME_WINDOW = config.TIME_WINDOW
OVERLAP = config.OVERLAP

PURE_STATES = ["bound", "confined", "transport", "exploratory"]
BEHAVIOR_STATE_ORDER = PURE_STATES.copy()
BEHAVIOR_STATE_PALETTE = {
    "bound": "#1b9e77",
    "confined": "#d95f02",
    "transport": "#7570b3",
    "exploratory": "#e7298a",
}
DECISION_PALETTE = {
    "high_confidence_single": "#0072B2",
    "boundary_like_pair": "#E69F00",
    "unresolved_low_confidence": "#999999",
}

# Paper switches. Set False to skip a heavy panel.
RUN_FIG1_TRACKS = True
RUN_FIG1_TOKENS = True
RUN_FIG1_EMBEDDING = True
RUN_FIG1_PROBS = True
RUN_FIG2_CONFUSION = True
RUN_FIG2_UNCERTAINTY = True
RUN_FIG2_FINGERPRINTS = True
RUN_FIG2_SUPERWINDOW = True
EMBED_SAMPLE_N = 80_000
EMBED_METHODS = ["umap", "phate"]  # both by default
EMBED_EXPORT_FORMAT = "png"  # PNG: UMAP/PHATE have too many points for Illustrator SVG
EMBED_DPI = 400  # high-resolution PNG
EMBED_PLOT_IN = 4.6  # square axes size in inches; legend does not change this
TRACK_EXAMPLES_PER_STATE = 6
# Shared micron box for every example window. Bound stays small; transport stays
# large. Raise this if long transport windows clip.
GALLERY_XY_SPAN_UM = 3.0
GALLERY_SCALE_BAR_UM = 0.5
GALLERY_TRACK_SIZE_UM = 3.0
# Fig 1e: along-track stacked softmax (smooth area or discrete bars)
F1E_SHOW_TRACK = True
F1E_SMOOTH = True  # True = curved stacked area; False = discrete stacked bars
F1E_SMOOTH_N = 400
F1E_MIN_WINDOWS = 8
F1E_PREFER_SWITCHES = True  # pick a continuous track that actually changes state
F1E_TIME_BAR_S = 2.0
RUN_FIG1E_EXTRA = True
RANDOM_SEED = 42

print("MODEL_VARIANT:", MODEL_VARIANT, "—", VARIANT["label"])
print("feature_mode:", VARIANT["feature_mode"])
print("Windowed predictions:", TRANSFORMER_WINDOWED_PARQUET)
print("Results:", RESULTS_DIR)

In [ ]:
# Shared helpers
def save_fig(fig, path, *, close=True, dpi=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    suffix = path.suffix.lower()
    if dpi is None:
        dpi = int(EMBED_DPI) if suffix in {".png", ".jpg", ".jpeg", ".tif", ".tiff"} else 300
    fig.savefig(path, dpi=dpi, bbox_inches="tight", transparent=True, facecolor="none")
    print("Saved:", path)
    if close:
        plt.close(fig)


def present(df, cols):
    return [c for c in cols if c in df.columns]


def high_conf_pure(df, state_col="transformer_behavior_state"):
    out = df.copy()
    if "is_high_confidence" in out.columns:
        out = out.loc[out["is_high_confidence"].astype(bool)]
    if state_col in out.columns:
        out = out.loc[out[state_col].isin(PURE_STATES)]
    return out


def emb_cols(df):
    return [c for c in df.columns if str(c).startswith("emb_")]


def _hex_to_rgb(h):
    h = str(h).lstrip("#")
    return tuple(int(h[i:i + 2], 16) for i in (0, 2, 4))


def _rgb_to_hex(rgb):
    return "#{:02x}{:02x}{:02x}".format(*(max(0, min(255, int(v))) for v in rgb))


def _mix_hex(a, b, t=0.5):
    ra, ga, ba = _hex_to_rgb(a)
    rb, gb, bb = _hex_to_rgb(b)
    return _rgb_to_hex((ra + (rb - ra) * t, ga + (gb - ga) * t, ba + (bb - ba) * t))


def mixture_state_color(label):
    s = str(label)
    if s in BEHAVIOR_STATE_PALETTE:
        return BEHAVIOR_STATE_PALETTE[s]
    if s in DECISION_PALETTE:
        return DECISION_PALETTE[s]
    if s.startswith("unresolved_"):
        base = s[len("unresolved_"):]
        if base in BEHAVIOR_STATE_PALETTE:
            return _mix_hex(BEHAVIOR_STATE_PALETTE[base], "#d0d0d0", 0.45)
        return "#b0b0b0"
    parts = s.split("_")
    if len(parts) == 2 and parts[0] in BEHAVIOR_STATE_PALETTE and parts[1] in BEHAVIOR_STATE_PALETTE:
        return _mix_hex(BEHAVIOR_STATE_PALETTE[parts[0]], BEHAVIOR_STATE_PALETTE[parts[1]], 0.5)
    return "#9a9a9a"


def mixture_state_order(labels):
    seen = []
    for s in labels:
        s = str(s)
        if s not in seen:
            seen.append(s)
    pures = [s for s in PURE_STATES if s in seen]
    pairs = sorted(
        s for s in seen
        if s not in pures and not str(s).startswith("unresolved_") and s not in DECISION_PALETTE
    )
    unresolved = sorted(s for s in seen if str(s).startswith("unresolved_"))
    other = [s for s in seen if s not in pures and s not in pairs and s not in unresolved]
    return pures + pairs + unresolved + other


def all_assignment_labels():
    pairs = ["_".join(sorted((a, b))) for i, a in enumerate(PURE_STATES) for b in PURE_STATES[i + 1 :]]
    unresolved = [f"unresolved_{s}" for s in PURE_STATES]
    return list(PURE_STATES) + pairs + unresolved


def square_embed_axes(legend_w=2.8):
    # Square plot of fixed inch size. Legend sits in reserved right margin.
    plot_in = float(EMBED_PLOT_IN)
    pad_l, pad_b, pad_t = 0.72, 0.58, 0.52
    fig_w = pad_l + plot_in + 0.18 + float(legend_w)
    fig_h = pad_b + plot_in + pad_t
    fig = plt.figure(figsize=(fig_w, fig_h))
    ax = fig.add_axes([pad_l / fig_w, pad_b / fig_h, plot_in / fig_w, plot_in / fig_h])
    ax.set_box_aspect(1)
    return fig, ax

## 1. Load held-out transformer windows

Window-level parquet already contains:

- certain / hierarchical-gate labels (`final_population`, `final_population_certain`)
- softmax 4-tuple (`prob_bound`, `prob_confined`, `prob_exploratory`, `prob_transport`)
- 16-D embedding (`emb_00` … `emb_15`)
- mixture reading (`transformer_behavior_state`, `decision_type`, `entropy`, …)

Instant rows are loaded only when a panel needs XY tracks.

In [ ]:
windowed_pl = pl.read_parquet(TRANSFORMER_WINDOWED_PARQUET)
if VARIANT_FILTER_RUN_ID is not None and "run_id" in windowed_pl.columns:
    windowed_pl = windowed_pl.filter(pl.col("run_id").cast(pl.Utf8) == str(VARIANT_FILTER_RUN_ID))
    print("Filtered to run_id:", VARIANT_FILTER_RUN_ID)
if "window_uid" in windowed_pl.columns:
    windowed_pl = windowed_pl.unique(subset=["window_uid"], keep="first")

DROP_UNGATED = True
UNGATED = ["ungatedgate1", "ungated", "unknown"]
if DROP_UNGATED:
    for col in present(windowed_pl, ["final_population", "final_population_certain", "transformer_behavior_state"]):
        windowed_pl = windowed_pl.filter(~pl.col(col).cast(pl.Utf8).is_in(UNGATED))

windowed_df = windowed_pl.to_pandas()
print("windowed_df:", windowed_df.shape)
print("columns of interest:")
display(pd.Series(present(windowed_df, [
    "window_uid", "unique_id", "mol", "geno", "type", "location",
    "final_population", "final_population_certain",
    "pred_label", "transformer_behavior_state", "decision_type",
    "prob_bound", "prob_confined", "prob_transport", "prob_exploratory",
    "top1_prob", "entropy", "entropy_norm", "n_valid_frames",
])))
print("\ntransformer_behavior_state counts:")
print(windowed_df["transformer_behavior_state"].value_counts(dropna=False))
if "mol" in windowed_df.columns:
    print("\nmol counts:")
    print(windowed_df["mol"].value_counts(dropna=False))
if "decision_type" in windowed_df.columns:
    print("\ndecision_type counts:")
    print(windowed_df["decision_type"].value_counts(dropna=False))

In [ ]:
# Optional instant load for track panels. Deferred until needed.
instant_df = None

def load_instant(max_uids=None, uids=None):
    global instant_df
    cols_keep = None
    scan = pl.scan_parquet(TRANSFORMER_INSTANT_PARQUET)
    names = scan.collect_schema().names()
    want = present(pd.DataFrame(columns=names), [
        "unique_id", "window_uid", "x_um", "y_um", "x", "y", "frame", "time_s",
        "direction_rad", "filename", "mol", "geno", "type", "location",
        "transformer_behavior_state", "pred_label", "final_population_certain",
        "prob_bound", "prob_confined", "prob_transport", "prob_exploratory",
    ])
    lf = scan.select(want)
    if uids is not None:
        uid_list = [str(u) for u in pd.Series(uids).astype(str).unique().tolist()]
        lf = lf.filter(pl.col("window_uid").cast(pl.Utf8).is_in(uid_list))
    elif max_uids is not None:
        keep = (
            windowed_df["window_uid"].astype(str)
            .drop_duplicates()
            .head(int(max_uids))
            .tolist()
        )
        lf = lf.filter(pl.col("window_uid").cast(pl.Utf8).is_in(keep))
    instant_df = lf.collect().to_pandas()
    print("instant_df:", instant_df.shape)
    return instant_df

## Figure 1 — Framework

### F1 schematic

The architecture drawing is finished in Illustrator. This cell writes a
machine-readable spec and a crude box diagram so the panel list is not lost.

In [ ]:
spec = {
    "run_id": "block3_long_large",
    "evaluation_dir": str(TRANSFORMER_BEHAVIOR_DIR),
    "inputs": ["dx", "dy", "direction_rad"],
    "feature_mode": "block3",
    "block_size": 3,
    "window_frames": TIME_WINDOW,
    "overlap_frames": OVERLAP,
    "dt_s": FRAME_INTERVAL,
    "pixel_size_um": PIXEL_SIZE_UM,
    "architecture": {
        "input_dim": 3,
        "embed_dim": 16,
        "num_heads": 4,
        "ff_dim": 256,
        "num_layers": 3,
        "pooling": "CLS",
        "head": "MLP 16-32-4 ReLU dropout 0.2",
    },
    "pretraining": {
        "objective": "InfoNCE",
        "temperature": 0.2,
        "augmentation": "shuffle_scale_angle (valid_prefix, segment=10)",
    },
    "finetuning": {
        "labels": "final_population_certain",
        "excluded": ["unknown", "ungatedgate1"],
        "loss": "cross-entropy",
        "subset_fraction": 0.10,
        "split": "lowtrain_hightest_20260616",
    },
    "states": PURE_STATES,
}
(FIG1_DIR / "F1a_architecture_spec.json").write_text(json.dumps(spec, indent=2), encoding="utf-8")

fig, ax = plt.subplots(figsize=(10.5, 2.6))
boxes = [
    (0.02, "Raw SPT\n(x, y, t)"),
    (0.18, "60-frame\nwindows"),
    (0.34, "Δx, Δy, θ\nblock3 tokens"),
    (0.50, "Transformer\nCLS, 16-D"),
    (0.66, "MLP head\n4 logits"),
    (0.82, "Softmax\nP(state)"),
]
for x, text in boxes:
    ax.add_patch(plt.Rectangle((x, 0.28), 0.14, 0.50, fill=False, lw=1.4))
    ax.text(x + 0.07, 0.53, text, ha="center", va="center", fontsize=8)
    if x < 0.80:
        ax.annotate("", xy=(x + 0.175, 0.53), xytext=(x + 0.145, 0.53),
                    arrowprops=dict(arrowstyle="->", lw=1.0))
ax.text(0.50, 0.12, "Contrastive pretraining (InfoNCE, T=0.2)  →  CE fine-tune on certain gates",
        ha="center", fontsize=8)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
ax.set_title("Fig 1a — replace this box diagram in Illustrator")
save_fig(fig, FIG1_DIR / "F1a_schematic_placeholder.svg")
print("Wrote", FIG1_DIR / "F1a_architecture_spec.json")

### F1 example tracks

Uses the same gallery rules as `plots.plot_tracks_grid_by_cluster` and the superwindow track galleries: each window is centered, and **every panel uses the same micron box** (`GALLERY_XY_SPAN_UM`). Bound stays small; transport stays large. Do not autoscale per panel.

In [ ]:
def window_xy_um(seg):
    if {"x_um", "y_um"}.issubset(seg.columns):
        return seg["x_um"].to_numpy(dtype=float), seg["y_um"].to_numpy(dtype=float)
    return (
        seg["x"].to_numpy(dtype=float) * PIXEL_SIZE_UM,
        seg["y"].to_numpy(dtype=float) * PIXEL_SIZE_UM,
    )


def plot_fixed_scale_state_gallery(
    instant_df,
    track_meta,
    *,
    state_col="transformer_behavior_state",
    xy_span_um=GALLERY_XY_SPAN_UM,
    scale_bar_um=GALLERY_SCALE_BAR_UM,
    examples_per_state=TRACK_EXAMPLES_PER_STATE,
):
    """One row per state, one column per example. Identical micron limits on every axis."""
    n_states = len(PURE_STATES)
    fig, axes = plt.subplots(
        n_states,
        examples_per_state,
        figsize=(1.55 * examples_per_state + 0.8, 1.55 * n_states + 0.4),
    )
    if n_states == 1:
        axes = np.array([axes])
    half = float(xy_span_um) / 2.0
    for i, state in enumerate(PURE_STATES):
        rows = track_meta.loc[track_meta[state_col] == state].reset_index(drop=True)
        for j in range(examples_per_state):
            ax = axes[i, j]
            ax.set_aspect("equal", adjustable="box")
            ax.set_xlim(-half, half)
            ax.set_ylim(-half, half)
            ax.set_xticks([])
            ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_visible(True)
                spine.set_linewidth(0.6)
                spine.set_color("0.35")
            if j >= len(rows):
                ax.axis("off")
                continue
            uid = rows.iloc[j]["window_uid"]
            seg = instant_df.loc[instant_df["window_uid"].astype(str) == str(uid)]
            if "frame" in seg.columns:
                seg = seg.sort_values("frame")
            if seg.empty:
                ax.axis("off")
                continue
            x, y = window_xy_um(seg)
            x = x - np.nanmean(x)
            y = y - np.nanmean(y)
            ax.plot(x, y, color=BEHAVIOR_STATE_PALETTE[state], lw=0.9, solid_capstyle="round")
            ax.scatter(x[0], y[0], s=7, c="k", zorder=3, linewidths=0)
            if j == 0:
                ax.set_ylabel(state, fontsize=10)
            if i == n_states - 1 and j == 0:
                x0, y0 = -half + 0.12 * xy_span_um, -half + 0.12 * xy_span_um
                ax.plot([x0, x0 + scale_bar_um], [y0, y0], color="k", lw=1.6, solid_capstyle="butt")
                ax.text(
                    x0 + scale_bar_um / 2,
                    y0 + 0.06 * xy_span_um,
                    f"{scale_bar_um:g} µm",
                    ha="center",
                    va="bottom",
                    fontsize=7,
                )
    fig.suptitle(
        f"Fig 1b — high-confidence windows, shared {xy_span_um:g} µm box",
        fontsize=11,
    )
    fig.tight_layout()
    return fig


if RUN_FIG1_TRACKS:
    need = high_conf_pure(windowed_df)
    rng = np.random.default_rng(RANDOM_SEED)
    picked = []
    for state in PURE_STATES:
        sub = need.loc[need["transformer_behavior_state"] == state]
        if sub.empty:
            print("No high-confidence windows for", state)
            continue
        take = min(TRACK_EXAMPLES_PER_STATE, len(sub))
        picked.append(sub.sample(n=take, random_state=int(rng.integers(0, 1_000_000))))
    track_meta = pd.concat(picked, ignore_index=True)
    print("Example windows:", track_meta.shape)
    inst = load_instant(uids=track_meta["window_uid"])
    if "x" not in inst.columns and {"x_um", "y_um"}.issubset(inst.columns):
        inst = inst.copy()
        inst["x"] = inst["x_um"] / PIXEL_SIZE_UM
        inst["y"] = inst["y_um"] / PIXEL_SIZE_UM

    fig = plot_fixed_scale_state_gallery(inst, track_meta)
    save_fig(fig, FIG1_DIR / "F1b_example_tracks.svg", close=False)
    track_meta.to_csv(FIG1_DIR / "F1b_example_window_uids.csv", index=False)

    # Same layout family as the figuremaker cluster galleries: one column per
    # state, fixed box per window, one global scale so sizes are comparable.
    try:
        plots.plot_tracks_grid_by_cluster(
            inst,
            cluster_col="transformer_behavior_state",
            segments_per_cluster=TRACK_EXAMPLES_PER_STATE,
            window_size=int(TIME_WINDOW),
            order=PURE_STATES,
            track_size_um=GALLERY_TRACK_SIZE_UM,
            grid_spacing_um=1.0,
            track_spacing_um=0.6,
            pixel_size_um=PIXEL_SIZE_UM,
            frame_interval=FRAME_INTERVAL,
            colorway="Dark2",
            figsize=(8.2, 9.0),
            line_thickness=0.7,
            transparent_background=True,
            save_path=str(FIG1_DIR),
            export_format="svg",
            show_plot=True,
            random_seed=RANDOM_SEED,
            show_cluster_labels=True,
            label_fontsize=11,
        )
    except Exception as exc:
        print("plot_tracks_grid_by_cluster skipped:", exc)
else:
    print("F1 tracks skipped")

### F1 motion tokens

Reconstructs the numbers the encoder actually sees for one example window.

**Top heatmap (raw tokens):** 3 rows × one column per frame. Row 0 is the x-step `Δx`, row 1 is the y-step `Δy`, row 2 is heading `θ` (`direction_rad`, not `Δθ`). A 60-frame window is a 3 × 60 matrix.

**Bottom heatmap (block3 tokens):** every 3 frames are collapsed to one token: `(ΣΔx, ΣΔy, circular-mean θ)`. 60 frames become 20 tokens. That is what `block3_long_large` is trained on.

The colour is the numeric value of that feature (coolwarm, not a state colour). The cell prints a short numeric diagnostic so the panel can be described in words.

In [ ]:
def tokens_from_xy(x, y, heading=None):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    dx = np.diff(x, prepend=x[0])
    dy = np.diff(y, prepend=y[0])
    if heading is None:
        th = np.arctan2(dy, dx)
    else:
        th = np.asarray(heading, dtype=float)
    th = np.nan_to_num(th, nan=0.0)
    raw = np.column_stack([dx, dy, th])
    rows = []
    for start in range(0, len(raw), 3):
        block = raw[start : start + 3]
        rows.append([
            block[:, 0].sum(),
            block[:, 1].sum(),
            np.arctan2(np.sin(block[:, 2]).mean(), np.cos(block[:, 2]).mean()),
        ])
    return raw, np.asarray(rows, dtype=float)


if RUN_FIG1_TOKENS:
    if instant_df is None:
        load_instant(uids=windowed_df["window_uid"].head(200))
    uid = None
    if (FIG1_DIR / "F1b_example_window_uids.csv").exists():
        meta = pd.read_csv(FIG1_DIR / "F1b_example_window_uids.csv")
        uid = str(meta.iloc[0]["window_uid"])
    if uid is None:
        uid = str(windowed_df["window_uid"].iloc[0])
    if instant_df is None or str(uid) not in set(instant_df["window_uid"].astype(str)):
        load_instant(uids=[uid])
    seg = instant_df.loc[instant_df["window_uid"].astype(str) == str(uid)].sort_values("frame")
    xcol = "x_um" if "x_um" in seg.columns else "x"
    ycol = "y_um" if "y_um" in seg.columns else "y"
    heading = seg["direction_rad"].to_numpy() if "direction_rad" in seg.columns else None
    raw, blocked = tokens_from_xy(seg[xcol], seg[ycol], heading)

    show_block = VARIANT.get("feature_mode", MODEL_VARIANT) == "block3"
    nrows = 2 if show_block else 1
    fig, axes = plt.subplots(nrows, 1, figsize=(7.2, 2.2 * nrows + 0.4), sharex=False, squeeze=False)
    im0 = axes[0, 0].imshow(raw.T, aspect="auto", interpolation="nearest", cmap="coolwarm")
    axes[0, 0].set_yticks([0, 1, 2], ["dx", "dy", "heading"])
    axes[0, 0].set_title(f"Raw tokens  window_uid={uid}  ({VARIANT['label']})")
    axes[0, 0].set_xlabel("frame in window")
    fig.colorbar(im0, ax=axes[0, 0], fraction=0.02)
    if show_block:
        im1 = axes[1, 0].imshow(blocked.T, aspect="auto", interpolation="nearest", cmap="coolwarm")
        axes[1, 0].set_yticks([0, 1, 2], ["sum dx", "sum dy", "mean heading"])
        axes[1, 0].set_title("block3 tokens (3-frame sums / circular mean heading)")
        axes[1, 0].set_xlabel("block index")
        fig.colorbar(im1, ax=axes[1, 0], fraction=0.02)
        np.save(FIG1_DIR / "F1c_block3_tokens.npy", blocked)
    fig.tight_layout()
    save_fig(fig, FIG1_DIR / "F1c_motion_tokens.svg")
    np.save(FIG1_DIR / "F1c_raw_tokens.npy", raw)

    # Numeric diagnostic: numbers you can turn into a figure legend in words.
    dt = float(FRAME_INTERVAL)
    n_raw = int(raw.shape[0])
    n_blk = int(blocked.shape[0])
    units = "um" if xcol == "x_um" else "px"
    dx, dy, th = raw[:, 0], raw[:, 1], raw[:, 2]
    step = np.hypot(dx, dy)
    print("=== Fig 1c token diagnostic ===")
    print(f"window_uid: {uid}")
    print(f"feature_mode: {VARIANT.get('feature_mode')}   ({VARIANT['label']})")
    print(f"n_frames: {n_raw}   n_block3_tokens: {n_blk}   expected_blocks: {int(np.ceil(n_raw / 3))}")
    print(f"dt: {dt:g} s   window_duration: {(n_raw - 1) * dt:.3f} s   block_duration: {3 * dt:.3f} s")
    print(f"units: {units}   heading: radians, direction_rad (theta, not dtheta)")
    print("raw rows: [0]=dx  [1]=dy  [2]=heading")
    print("block3 rows: [0]=sum(dx over 3 frames)  [1]=sum(dy)  [2]=circular mean heading")
    print(
        "raw dx:  "
        f"min={dx.min():.4f}  median={np.median(dx):.4f}  mean={dx.mean():.4f}  max={dx.max():.4f}  {units}/frame"
    )
    print(
        "raw dy:  "
        f"min={dy.min():.4f}  median={np.median(dy):.4f}  mean={dy.mean():.4f}  max={dy.max():.4f}  {units}/frame"
    )
    print(
        "step |dxy|:  "
        f"min={step.min():.4f}  median={np.median(step):.4f}  mean={step.mean():.4f}  max={step.max():.4f}  {units}"
    )
    print(
        "heading:  "
        f"min={th.min():.3f}  max={th.max():.3f} rad   "
        f"circ_mean={np.arctan2(np.sin(th).mean(), np.cos(th).mean()):.3f} rad"
    )
    print("First 6 raw tokens  [frame, dx, dy, theta_rad, step]:")
    for i in range(min(6, n_raw)):
        print(f"  {i:3d}  {dx[i]: .5f}  {dy[i]: .5f}  {th[i]: .4f}  {step[i]:.5f}")
    print("First 4 block3 tokens  [block, frames, sum_dx, sum_dy, circmean_theta, check_sum_dx]:")
    for j in range(min(4, n_blk)):
        a, b = 3 * j, min(3 * j + 3, n_raw)
        chk = float(dx[a:b].sum())
        print(
            f"  {j:3d}  frames {a}-{b - 1}  "
            f"{blocked[j, 0]: .5f}  {blocked[j, 1]: .5f}  {blocked[j, 2]: .4f}  "
            f"check_sum_dx={chk: .5f}"
        )
    print(
        "Net displacement of this window: "
        f"dx={dx.sum():.4f} {units}, dy={dy.sum():.4f} {units}, "
        f"|R|={np.hypot(dx.sum(), dy.sum()):.4f} {units}"
    )
    print("Colour in the heatmaps = those numeric values, not a behaviour state.")
else:
    print("F1 tokens skipped")

### F1 embedding

UMAP and PHATE of the 16-D CLS embedding. Default export is **high-resolution PNG** (`EMBED_DPI`). These clouds are too dense for Illustrator SVG.

Each colouring is a different reading of the same points:

| Colouring | What it is |
|---|---|
| predicted argmax (`pred_label`) | Hard 4-way call: the state with the largest softmax probability. Not an extra trained class. |
| certain gate (`final_population_certain`) | Curator certain-gate label used for fine-tune / held-out comparison. |
| mixture reading (`transformer_behavior_state`) | Post-hoc softmax rule: high-confidence single, boundary pair (e.g. `bound_confined`), or `unresolved_<top1>`. |
| decision type (`decision_type`) | Which rule fired: `high_confidence_single`, `boundary_like_pair`, or `unresolved_low_confidence`. |

Axes are a **fixed square**. The legend sits in a reserved right margin so longer label strings do not shrink the plot.

In [ ]:
if RUN_FIG1_EMBEDDING:
    cols = emb_cols(windowed_df)
    if len(cols) < 2:
        raise RuntimeError("No emb_* columns on windowed_df")
    plot_df = windowed_df.dropna(subset=cols).copy()
    if len(plot_df) > EMBED_SAMPLE_N:
        plot_df = plot_df.sample(EMBED_SAMPLE_N, random_state=RANDOM_SEED)
    color_jobs = [
        (
            "pred_label",
            "predicted argmax",
            "Hard call = argmax of the four softmax outputs (bound / confined / transport / exploratory).",
        ),
        (
            "final_population_certain",
            "certain gate",
            "Human/curator certain-gate label. This is the supervision target, not a model output.",
        ),
        (
            "transformer_behavior_state",
            "mixture reading",
            "Post-hoc reading of the same softmax: clean single, two-state boundary pair, or unresolved.",
        ),
        (
            "decision_type",
            "decision type",
            "Which mixture rule fired on that window (high-confidence / boundary-like / unresolved).",
        ),
    ]
    method_kwargs = {
        "umap": {"n_neighbors": 30, "min_dist": 0.1},
        "phate": {"knn": 15, "decay": 40, "t": "auto"},
    }
    print("Fig 1d colourings (same points, different labels):")
    for col, title, meaning in color_jobs:
        print(f"  {title:20s}  {col:28s}  {meaning}")
    for method in EMBED_METHODS:
        try:
            emb2 = evaluate.compute_embedding_2d(
                plot_df,
                emb_cols=cols,
                method=method,
                n_sample=None,
                random_seed=RANDOM_SEED,
                **method_kwargs.get(method, {}),
            )
        except Exception as exc:
            print(f"{method} skipped:", exc)
            continue
        xcol, ycol = f"{method}_1", f"{method}_2"
        for col, title, meaning in color_jobs:
            if col not in emb2.columns:
                continue
            fig, ax = square_embed_axes(legend_w=2.9)
            if col == "transformer_behavior_state":
                labels = emb2[col].dropna().astype(str).unique().tolist()
                hue_order = mixture_state_order(labels)
                palette = {s: mixture_state_color(s) for s in hue_order}
            elif col in ("pred_label", "final_population_certain"):
                hue_order = BEHAVIOR_STATE_ORDER
                palette = BEHAVIOR_STATE_PALETTE
            elif col == "decision_type":
                hue_order = list(DECISION_PALETTE)
                palette = DECISION_PALETTE
            else:
                hue_order = None
                palette = None
            sns.scatterplot(
                data=emb2, x=xcol, y=ycol, hue=col, hue_order=hue_order,
                palette=palette, s=4, alpha=0.35, linewidth=0, ax=ax, rasterized=True,
                legend="full",
            )
            ax.set_title(f"Fig 1d — {method.upper()} {VARIANT['feature_mode']}\n{title}", fontsize=10)
            ax.set_xlabel(xcol)
            ax.set_ylabel(ycol)
            ax.set_box_aspect(1)
            leg = ax.legend(
                bbox_to_anchor=(1.04, 1.0), loc="upper left",
                markerscale=3, fontsize=7, frameon=False, borderaxespad=0.0,
            )
            if leg is not None:
                leg.set_title(col, prop={"size": 7})
            save_fig(fig, FIG1_DIR / f"F1d_{method}_by_{col}.{EMBED_EXPORT_FORMAT}")
        emb2.head(5000).to_csv(FIG1_DIR / f"F1d_{method}_sample.csv", index=False)
else:
    print("F1 embedding skipped")

### F1 probabilities / intermediate prediction

Settings live in the **Fig 1e knobs cell**: unit (whole track vs 10/20-window superwindow), unresolved/length filters, content (`mixed` / a named state), **`F1E_RANDOM_SEED`**, and movie/smoothing. Change the seed to draw another example from the filtered pool. Left panels share one rectangle size (`F1E_LEFT_BOX_ASPECT`). The movie uses the same unit and the same `F1E_TRACK_SMOOTH` / `F1E_TRACK_CURVE` as the static XY trail.

One figure, four subplots plus a legend:

- **Top left:** stacked softmax vs time (`F1E_SMOOTH` toggles curved area vs discrete bars). Colours are the four trained states.
- **Middle left:** blob sequence of `transformer_behavior_state`, including boundary pairs and unresolved labels.
- **Bottom left:** normalized entropy of that same softmax, aligned in time.
- **Top right:** legend for every possible assignment (4 pures + 6 pairs + 4 unresolved).
- **Right, spanning the lower rows:** the same track in XY, coloured by that assignment.

Normalized entropy is Shannon entropy of the 4-way softmax, divided by `log2(4)` so it runs from 0 (one state has probability 1) to 1 (all four probabilities are 0.25). High values mean the model is split; low values mean a confident argmax.

Fig 1e extra keeps the window-to-window fan for this track.

In [ ]:
# =============================================================================
# Fig 1e settings — edit here, then run this cell and the next
# =============================================================================

# --- display ---
F1E_SMOOTH = True                  # True = curved stacked area; False = discrete bars
F1E_SMOOTH_N = 400
F1E_TIME_BAR_S = 2.0
F1E_LEFT_BOX_ASPECT = 0.38         # height/width of EACH left panel (all three match)
F1E_BLOB_Y_MODE = "observed"       # "observed" rows, or "all" 14 assignment labels
RUN_FIG1E_EXTRA = True

# --- unit ---
# "track" = whole unique_id.
# "superwindow" = L consecutive transformer windows (60 frames, 30 overlap each).
# L=10 or 20 matches the plasticity notebooks. This is NOT L raw frames.
F1E_UNIT = "track"                 # "track" or "superwindow"
F1E_SUPERWINDOW_LENGTH = 10        # 10 or 20; ignored when UNIT == "track"

# --- automatic quality / length filter ---
F1E_REQUIRE_CONTINUOUS = True      # no missing windows inside the unit
F1E_EXCLUDE_UNRESOLVED = True      # drop units that contain any unresolved_*
F1E_MIN_WINDOWS = 8                # min transformer windows in the unit
F1E_MAX_WINDOWS = 60               # None = no maximum

# --- content filter (pick ONE) ---
# "mixed" | "bound" | "confined" | "transport" | "exploratory" | "any"
F1E_CONTENT = "mixed"
F1E_MIN_TARGET_FRACTION = 0.35     # named state: minimum fraction of windows
F1E_MIN_DISTINCT_FOR_MIXED = 3     # mixed: at least this many distinct assignments
F1E_MIN_SWITCHES_FOR_MIXED = 3     # mixed: at least this many assignment changes

# =============================================================================
# RANDOMIZER — change the seed to draw a different example from the filtered pool
# =============================================================================
F1E_RANDOM_SEED = 7
F1E_PICK_AMONG_TOP = 12            # shuffle this many top-scoring units, take one
F1E_FORCE_UNIQUE_ID = None         # set to a unique_id string to override filters

# --- movie: same unit and same length as the figure ---
# Uses movies.render_state_change_track_movie (head_pulse / track_gold flashes).
F1E_MAKE_MOVIE = False             # True also writes the Fig 1e movie (slow)
F1E_OVERLAY_ON_RAW = False         # True only if the raw TIFF can be found
F1E_FLASH_MODES = ("head_pulse", "track_gold")  # add "lut_whiten" if you want
F1E_FLASH_HOLD_FRAMES = 6
F1E_TRACK_SMOOTH = 1.2             # Gaussian sigma in frames; 0 = off
F1E_TRACK_CURVE = 4                # cubic samples/segment; 1 = straight chords
F1E_TRACK_LINEWIDTH = 2.4          # also used for the static XY panel
F1E_MOVIE_FPS = 16
F1E_MOVIE_BACKGROUND = "white"

print("Fig 1e knobs loaded")
print(f"  UNIT={F1E_UNIT}  SW_LEN={F1E_SUPERWINDOW_LENGTH}  CONTENT={F1E_CONTENT}")
print(f"  SEED={F1E_RANDOM_SEED}   <-- change F1E_RANDOM_SEED to pick another example")
print(f"  movie={F1E_MAKE_MOVIE}  track_smooth={F1E_TRACK_SMOOTH}  track_curve={F1E_TRACK_CURVE}")

In [ ]:
def window_times_s(seq):
    step_s = max(1, int(TIME_WINDOW) - int(OVERLAP)) * float(FRAME_INTERVAL)
    if "frame_start" in seq.columns:
        t = pd.to_numeric(seq["frame_start"], errors="coerce").to_numpy(dtype=float) * float(FRAME_INTERVAL)
        if np.isfinite(t).any():
            return t, step_s
    if "time_window" in seq.columns:
        tw = pd.to_numeric(seq["time_window"], errors="coerce").to_numpy(dtype=float)
        if np.isfinite(tw).any():
            return tw * step_s, step_s
    return np.arange(len(seq), dtype=float) * step_s, step_s


def add_time_bar(ax, t, bar_s, y=-0.12):
    t0 = float(np.nanmin(t))
    t1 = t0 + float(bar_s)
    ax.plot([t0, t1], [y, y], color="k", lw=2.2, solid_capstyle="butt", clip_on=False)
    ax.text((t0 + t1) / 2, y - 0.04, f"{bar_s:g} s", ha="center", va="top", fontsize=8, clip_on=False)


def softmax_stack(seq):
    mats = []
    for state in PURE_STATES:
        col = f"prob_{state}"
        if col in seq.columns:
            mats.append(pd.to_numeric(seq[col], errors="coerce").to_numpy(dtype=float))
        else:
            mats.append(np.zeros(len(seq), dtype=float))
    P = np.nan_to_num(np.vstack(mats), nan=0.0)
    tot = P.sum(axis=0)
    tot = np.where(tot > 0, tot, 1.0)
    return P / tot


def interpolate_stack(t, P, n=400):
    t = np.asarray(t, dtype=float)
    order = np.argsort(t)
    t, P = t[order], P[:, order]
    _, uniq = np.unique(t, return_index=True)
    t, P = t[uniq], P[:, uniq]
    t_fine = np.linspace(float(t.min()), float(t.max()), int(max(n, len(t))))
    try:
        from scipy.interpolate import PchipInterpolator
        P_fine = np.vstack([PchipInterpolator(t, P[i])(t_fine) for i in range(P.shape[0])])
    except Exception:
        P_fine = np.vstack([np.interp(t_fine, t, P[i]) for i in range(P.shape[0])])
    P_fine = np.clip(P_fine, 0.0, None)
    tot = P_fine.sum(axis=0)
    tot = np.where(tot > 0, tot, 1.0)
    return t_fine, P_fine / tot


def plot_stacked_softmax(ax, t, P, step_s, smooth):
    colors = [BEHAVIOR_STATE_PALETTE[s] for s in PURE_STATES]
    if (not smooth) or (len(t) < 3):
        bottoms = np.zeros(len(t))
        for i, state in enumerate(PURE_STATES):
            ax.bar(
                t, P[i], bottom=bottoms, color=colors[i], width=step_s,
                align="edge", linewidth=0, label=state,
            )
            bottoms = bottoms + P[i]
    else:
        t_fine, P_fine = interpolate_stack(t, P, n=int(F1E_SMOOTH_N))
        ax.stackplot(t_fine, P_fine, colors=colors, labels=list(PURE_STATES), alpha=0.92, linewidth=0)
        tops = np.cumsum(P_fine, axis=0)
        for i, color in enumerate(colors):
            ax.plot(t_fine, tops[i], color=color, lw=1.5, solid_capstyle="round")
    ax.set_ylim(0, 1)
    ax.set_ylabel("P(state)")


def track_is_continuous(g):
    # True when windows form one unbroken run (no dropped/missing windows).
    sort_col = "time_window" if "time_window" in g.columns else ("frame_start" if "frame_start" in g.columns else None)
    if sort_col is None:
        return False
    g = g.sort_values(sort_col)
    if sort_col == "time_window":
        tw = pd.to_numeric(g["time_window"], errors="coerce").to_numpy(dtype=float)
        return len(tw) >= 2 and np.isfinite(tw).all() and bool(np.allclose(np.diff(tw), 1.0))
    expected_fr = float(max(1, int(TIME_WINDOW) - int(OVERLAP)))
    fr = pd.to_numeric(g["frame_start"], errors="coerce").to_numpy(dtype=float)
    return len(fr) >= 2 and np.isfinite(fr).all() and bool(np.allclose(np.diff(fr), expected_fr))


def _assignment_stats(states):
    states = pd.Series(states, dtype=str)
    n = int(len(states))
    n_distinct = int(states.nunique())
    n_switch = int((states != states.shift()).sum() - (0 if n == 0 else 1))
    vc = states.value_counts(normalize=True)
    p = vc.to_numpy(dtype=float)
    entropy = float(-(p * np.log(p + 1e-12)).sum()) if n else 0.0
    stats = {
        "n_windows": n,
        "n_distinct": n_distinct,
        "n_switches": max(n_switch, 0),
        "entropy": entropy,
        "mix_score": max(n_switch, 0) * (1.0 + entropy),
        "n_unresolved": int(states.str.startswith("unresolved_").sum()),
        "has_unresolved": bool(states.str.startswith("unresolved_").any()),
    }
    for s in PURE_STATES:
        stats[f"frac_{s}"] = float((states == s).mean()) if n else 0.0
        stats[f"mention_{s}"] = float(states.map(lambda x, lab=s: lab in str(x).replace("unresolved_", "").split("_") or str(x) == lab).mean()) if n else 0.0
    return stats


def iter_track_groups(df):
    id_cols = [c for c in ("filename", "unique_id") if c in df.columns]
    if "unique_id" not in id_cols:
        raise KeyError("unique_id is required to pick a Fig 1e unit")
    for key, g in df.dropna(subset=["unique_id"]).groupby(id_cols, dropna=False):
        key = key if isinstance(key, tuple) else (key,)
        uid = key[id_cols.index("unique_id")]
        fn = key[id_cols.index("filename")] if "filename" in id_cols else None
        sort_col = "time_window" if "time_window" in g.columns else g.columns[0]
        yield uid, fn, g.sort_values(sort_col)


def score_unit_row(uid, fn, start_i, sub):
    states = sub["transformer_behavior_state"] if "transformer_behavior_state" in sub.columns else pd.Series(dtype=str)
    stats = _assignment_stats(states)
    stats.update({
        "uid": uid,
        "filename": fn,
        "start_i": start_i,
        "end_i": int(start_i) + int(len(sub)) - 1 if start_i is not None else int(len(sub)) - 1,
    })
    return stats


def build_f1e_candidate_pool(df):
    unit = str(F1E_UNIT).strip().lower()
    sw_len = int(F1E_SUPERWINDOW_LENGTH)
    rows = []
    n_groups = 0
    for uid, fn, g in iter_track_groups(df):
        n_groups += 1
        if F1E_FORCE_UNIQUE_ID is not None and str(uid) != str(F1E_FORCE_UNIQUE_ID):
            continue
        continuous = track_is_continuous(g)
        if unit == "superwindow":
            if F1E_REQUIRE_CONTINUOUS and not continuous:
                continue
            if len(g) < sw_len:
                continue
            for start in range(0, len(g) - sw_len + 1):
                sub = g.iloc[start:start + sw_len]
                if F1E_REQUIRE_CONTINUOUS and "time_window" in sub.columns:
                    tw = pd.to_numeric(sub["time_window"], errors="coerce").to_numpy(dtype=float)
                    if not np.allclose(np.diff(tw), 1.0):
                        continue
                rows.append(score_unit_row(uid, fn, start, sub))
        else:
            if F1E_REQUIRE_CONTINUOUS and not continuous:
                continue
            rows.append(score_unit_row(uid, fn, 0, g))
    pool = pd.DataFrame(rows)
    print(f"Fig 1e pool: grouped {n_groups} tracks; built {len(pool)} {unit} units")
    return pool


def filter_f1e_candidates(pool):
    df = pool.copy()
    if df.empty:
        return df
    if F1E_FORCE_UNIQUE_ID is not None:
        print(f"FORCE unique_id={F1E_FORCE_UNIQUE_ID}: skipping content filters")
        return df.reset_index(drop=True)
    n0 = len(df)
    if F1E_EXCLUDE_UNRESOLVED and "has_unresolved" in df.columns:
        df = df.loc[~df["has_unresolved"]].copy()
    df = df.loc[df["n_windows"] >= int(F1E_MIN_WINDOWS)].copy()
    if F1E_MAX_WINDOWS is not None:
        df = df.loc[df["n_windows"] <= int(F1E_MAX_WINDOWS)].copy()
    content = str(F1E_CONTENT).strip().lower()
    if content in PURE_STATES:
        df = df.loc[df[f"frac_{content}"] >= float(F1E_MIN_TARGET_FRACTION)].copy()
        df = df.sort_values([f"frac_{content}", "n_windows"], ascending=False)
    elif content == "mixed":
        df = df.loc[
            (df["n_distinct"] >= int(F1E_MIN_DISTINCT_FOR_MIXED))
            & (df["n_switches"] >= int(F1E_MIN_SWITCHES_FOR_MIXED))
        ].copy()
        df = df.sort_values(["mix_score", "n_distinct", "n_switches"], ascending=False)
    else:
        df = df.sort_values(["n_windows", "mix_score"], ascending=False)
    print(
        f"Fig 1e filters: {n0} → {len(df)} after unresolved/length/content={content} "
        f"(min_w={F1E_MIN_WINDOWS}, max_w={F1E_MAX_WINDOWS})"
    )
    return df.reset_index(drop=True)


def pick_f1e_unit(df):
    pool = build_f1e_candidate_pool(df)
    filtered = filter_f1e_candidates(pool)
    if filtered.empty:
        print("No unit survived filters; falling back to the longest track. Relax F1E_CONTENT / length / unresolved knobs.")
        longest = None
        for uid, fn, g in iter_track_groups(df):
            if longest is None or len(g) > longest[2]:
                longest = (uid, fn, len(g), g)
        uid, fn, _, g = longest
        return uid, fn, g, filtered, False
    n_top = max(1, min(int(F1E_PICK_AMONG_TOP), len(filtered)))
    top = filtered.head(n_top)
    rng = np.random.default_rng(int(F1E_RANDOM_SEED))
    pick_i = int(rng.integers(0, len(top)))
    chosen = top.iloc[pick_i]
    uid, fn = chosen["uid"], chosen["filename"]
    start_i = int(chosen["start_i"])
    n_w = int(chosen["n_windows"])
    seq_mask = df["unique_id"] == uid
    if fn is not None and "filename" in df.columns:
        seq_mask = seq_mask & (df["filename"] == fn)
    g = df.loc[seq_mask].sort_values("time_window" if "time_window" in df.columns else df.columns[0])
    if str(F1E_UNIT).strip().lower() == "superwindow":
        g = g.iloc[start_i:start_i + n_w]
    print(
        f"PICK  seed={F1E_RANDOM_SEED}  among_top={n_top}  index={pick_i}  "
        f"uid={uid}  n_windows={n_w}  content={F1E_CONTENT}"
    )
    print(top.head(8).to_string(index=False))
    return uid, fn, g, filtered, True


def plot_state_blobs_on_ax(ax, t, states, state_order, colors):
    ymap = {str(s): i for i, s in enumerate(state_order)}
    ys = np.array([ymap.get(str(s), np.nan) for s in states], dtype=float)
    t = np.asarray(t, dtype=float)
    for i, (ti, yi, s) in enumerate(zip(t, ys, states)):
        if not np.isfinite(yi):
            continue
        ax.scatter(
            ti, yi, s=86, c=[colors.get(str(s), "#aaaaaa")],
            edgecolors="k", linewidths=0.75, zorder=5,
        )
        if i + 1 < len(t) and np.isfinite(ys[i + 1]):
            ax.plot([ti, t[i + 1]], [yi, ys[i + 1]], color="k", lw=1.05, alpha=0.45, zorder=3)
    ax.set_yticks(range(len(state_order)))
    ax.set_yticklabels(state_order, fontsize=7)
    ax.set_ylim(-0.7, len(state_order) - 0.3)
    ax.set_xlabel("time (s)")
    ax.grid(True, axis="x", alpha=0.2)


def plot_colored_xy_on_ax(ax, inst, state_col, colors, span_um, scale_bar_um, track_smooth=0.0, track_curve=1, linewidth=2.4):
    sort_cols = [c for c in ("frame", "time_s") if c in inst.columns]
    if sort_cols:
        inst = inst.sort_values(sort_cols)
    x = pd.to_numeric(inst["x_um"], errors="coerce").to_numpy(dtype=float)
    y = pd.to_numeric(inst["y_um"], errors="coerce").to_numpy(dtype=float)
    states = inst[state_col].astype(str).to_numpy()
    tmp = pd.DataFrame({"x": x, "y": y, "state": states})
    tmp = movies._smooth_track_coordinates(tmp, smooth=track_smooth)
    xs, ys, sts = movies._curved_polyline(
        tmp["x"].to_numpy(dtype=float),
        tmp["y"].to_numpy(dtype=float),
        tmp["state"].to_numpy(),
        samples_per_seg=int(track_curve),
    )
    for i in range(max(0, len(xs) - 1)):
        ax.plot(
            xs[i:i + 2], ys[i:i + 2],
            color=colors.get(str(sts[i]), "lightgray"),
            lw=linewidth, solid_capstyle="round", solid_joinstyle="round",
        )
    ax.scatter(xs, ys, s=4, color=[colors.get(str(s), "lightgray") for s in sts], edgecolor="none", zorder=4)
    xm = 0.5 * (np.nanmin(xs) + np.nanmax(xs))
    ym = 0.5 * (np.nanmin(ys) + np.nanmax(ys))
    span = float(span_um)
    ax.set_xlim(xm - 0.5 * span, xm + 0.5 * span)
    ax.set_ylim(ym - 0.5 * span, ym + 0.5 * span)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xticks([])
    ax.set_yticks([])
    x0 = xm - 0.5 * span + 0.08 * span
    y0 = ym - 0.5 * span + 0.08 * span
    ax.plot([x0, x0 + scale_bar_um], [y0, y0], color="k", lw=2.2, solid_capstyle="butt")
    ax.text(x0 + 0.5 * scale_bar_um, y0 + 0.04 * span, f"{scale_bar_um:g} um", ha="center", va="bottom", fontsize=7)


def draw_assignment_legend(ax, labels, colors):
    ax.set_axis_off()
    ax.set_title("All assignments", fontsize=10)
    pures = [s for s in labels if s in PURE_STATES]
    pairs = [s for s in labels if s not in PURE_STATES and not str(s).startswith("unresolved_")]
    unresolved = [s for s in labels if str(s).startswith("unresolved_")]
    y = 0.96
    for title, group in (
        ("High-confidence states", pures),
        ("Boundary / transition pairs", pairs),
        ("Unresolved (low confidence)", unresolved),
    ):
        ax.text(0.04, y, title, transform=ax.transAxes, fontsize=8, fontweight="bold", va="top")
        y -= 0.055
        for s in group:
            ax.scatter(
                [0.10], [y], s=90, c=[colors.get(s, "#aaaaaa")],
                edgecolors="k", linewidths=0.7, transform=ax.transAxes, clip_on=False,
            )
            ax.text(0.18, y, s, transform=ax.transAxes, fontsize=7.5, va="center")
            y -= 0.052
        y -= 0.02
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)


if RUN_FIG1_PROBS:
    need_cols = present(windowed_df, [
        "unique_id", "window_uid", "time_window", "frame_start", "pred_label", "transformer_behavior_state",
        "decision_type", "prob_bound", "prob_confined", "prob_transport", "prob_exploratory",
        "top1_prob", "entropy_norm",
    ])
    pick_id, pick_fn, seq, f1e_pool, found_ok = pick_f1e_unit(windowed_df)
    F1E_PICK_ID = pick_id
    F1E_PICK_FILENAME = pick_fn
    F1E_SEQ = seq
    t, step_s = window_times_s(seq)
    tw_step = np.diff(pd.to_numeric(seq["time_window"], errors="coerce")) if "time_window" in seq.columns else None
    n_gaps = int(np.sum(~np.isclose(tw_step, 1.0))) if tw_step is not None and len(tw_step) else 0
    P = softmax_stack(seq)
    assign_labels = all_assignment_labels()
    assign_colors = {s: mixture_state_color(s) for s in assign_labels}
    state_col = "transformer_behavior_state"
    state_seq = seq[state_col].astype(str).tolist() if state_col in seq.columns else []
    blob_order = assign_labels if str(F1E_BLOB_Y_MODE).strip().lower() == "all" else mixture_state_order(state_seq)

    fig = plt.figure(figsize=(12.4, 11.4))
    gs = fig.add_gridspec(
        3, 2, width_ratios=[1.55, 1.0], height_ratios=[1.0, 1.0, 1.0],
        wspace=0.28, hspace=0.30,
    )
    ax_soft = fig.add_subplot(gs[0, 0])
    ax_blob = fig.add_subplot(gs[1, 0], sharex=ax_soft)
    ax_ent = fig.add_subplot(gs[2, 0], sharex=ax_soft)
    ax_leg = fig.add_subplot(gs[0, 1])
    ax_xy = fig.add_subplot(gs[1:, 1])

    plot_stacked_softmax(ax_soft, t, P, step_s, F1E_SMOOTH)
    style = "smooth stacked area" if F1E_SMOOTH else "stacked bars"
    ax_soft.set_title(f"Fig 1e — softmax vs time ({style})")
    ax_soft.tick_params(labelbottom=False)
    ax_soft.set_xlabel("")

    plot_state_blobs_on_ax(ax_blob, t, state_seq, blob_order, assign_colors)
    ax_blob.set_title("State assignment blobs (including transitions)")
    ax_blob.tick_params(labelbottom=False)
    ax_blob.set_xlabel("")

    if "entropy_norm" in seq.columns:
        ent = pd.to_numeric(seq["entropy_norm"], errors="coerce").to_numpy(dtype=float)
        if F1E_SMOOTH and len(t) >= 3:
            t_fine, _ = interpolate_stack(t, P, n=int(F1E_SMOOTH_N))
            order = np.argsort(t)
            tt, ee = np.asarray(t, dtype=float)[order], ent[order]
            _, uniq = np.unique(tt, return_index=True)
            ax_ent.plot(t_fine, np.interp(t_fine, tt[uniq], ee[uniq]), color="k", lw=1.4)
        else:
            ax_ent.plot(t, ent, color="k", lw=1.3)
        print(
            "Normalized entropy (0=one state certain, 1=uniform 0.25 each): "
            f"min={np.nanmin(ent):.3f}  median={np.nanmedian(ent):.3f}  "
            f"mean={np.nanmean(ent):.3f}  max={np.nanmax(ent):.3f}"
        )
    ax_ent.set_ylim(0, 1)
    ax_ent.set_ylabel("normalized entropy")
    ax_ent.set_xlabel("time (s)")
    ax_ent.set_title("Softmax entropy")
    add_time_bar(ax_ent, t, F1E_TIME_BAR_S)

    for ax_left in (ax_soft, ax_blob, ax_ent):
        ax_left.set_box_aspect(float(F1E_LEFT_BOX_ASPECT))

    draw_assignment_legend(ax_leg, assign_labels, assign_colors)

    inst = None
    if "window_uid" not in seq.columns:
        ax_xy.set_axis_off()
        ax_xy.text(0.5, 0.5, "No window_uid; cannot draw XY", ha="center", va="center")
    else:
        inst = load_instant(uids=seq["window_uid"])
        inst = inst.copy()
        if state_col not in inst.columns or inst[state_col].isna().all():
            inst = inst.merge(
                seq[["window_uid", state_col]].drop_duplicates("window_uid"),
                on="window_uid", how="left",
            )
        if "x_um" not in inst.columns:
            if "x" not in inst.columns:
                raise KeyError("instant table needs x_um or x")
            inst["x_um"] = pd.to_numeric(inst["x"], errors="coerce") * float(PIXEL_SIZE_UM)
            inst["y_um"] = pd.to_numeric(inst["y"], errors="coerce") * float(PIXEL_SIZE_UM)
        sort_cols = [c for c in ("frame", "time_s") if c in inst.columns]
        if sort_cols:
            inst = inst.sort_values(sort_cols)
        if "frame" in inst.columns:
            inst = inst.drop_duplicates("frame", keep="last")
        inst["unique_id"] = pick_id
        plot_colored_xy_on_ax(
            ax_xy, inst, state_col, assign_colors,
            span_um=GALLERY_XY_SPAN_UM, scale_bar_um=GALLERY_SCALE_BAR_UM,
            track_smooth=F1E_TRACK_SMOOTH, track_curve=F1E_TRACK_CURVE,
            linewidth=F1E_TRACK_LINEWIDTH,
        )
        ax_xy.set_title(f"Track coloured by assignment  (smooth={F1E_TRACK_SMOOTH:g}, curve={F1E_TRACK_CURVE})")

    fig.suptitle(
        f"Fig 1e   {VARIANT['feature_mode']}   {F1E_UNIT}   unique_id={pick_id}   n={len(seq)}",
        fontsize=11, y=0.995,
    )
    tag = "smooth" if F1E_SMOOTH else "bars"
    save_fig(fig, FIG1_DIR / f"F1e_softmax_{VARIANT['feature_mode']}_{tag}.svg", close=False)
    plt.show()

    seq[need_cols].to_csv(FIG1_DIR / "F1e_track_softmax.csv", index=False)
    print("unique_id:", pick_id, "  unit:", F1E_UNIT, "  windows:", len(seq), "  gaps:", n_gaps)
    print("transformer_behavior_state:")
    print(seq[state_col].value_counts(dropna=False) if state_col in seq else "n/a")
    print("Time step between windows:", f"{step_s:g} s;", f"duration {float(np.nanmax(t) - np.nanmin(t)):.2f} s")

    if F1E_MAKE_MOVIE:
        if inst is None or inst.empty:
            print("Movie skipped: no instant localizations for this unit.")
        else:
            movie_dir = FIG1_DIR / "F1e_state_change_movie" / str(pick_id).replace("/", "_")
            movie = movies.render_state_change_track_movie(
                inst,
                pick_id,
                view_mode="whole_track",
                track_id_col="unique_id",
                state_col=state_col,
                state_colors=assign_colors,
                state_order=assign_labels,
                overlay_on_raw=F1E_OVERLAY_ON_RAW,
                background=F1E_MOVIE_BACKGROUND,
                flash_modes=F1E_FLASH_MODES,
                flash_hold_frames=F1E_FLASH_HOLD_FRAMES,
                track_smooth=F1E_TRACK_SMOOTH,
                track_curve=F1E_TRACK_CURVE,
                track_linewidth=F1E_TRACK_LINEWIDTH,
                trail="cumulative",
                pixel_size_um=PIXEL_SIZE_UM,
                frame_interval=FRAME_INTERVAL,
                write_gif=True,
                write_png=True,
                fps=F1E_MOVIE_FPS,
                preview_frames=True,
                show_plot=False,
                output_dir=movie_dir,
            )
            print("Movie PNG dir:", movie["output_dir"])
            print("Movie GIF:", movie.get("gif_path"))
            print("Movie frames:", movie["n_frames"], "  flashes/state-changes:", movie["n_state_changes"])
            try:
                from IPython.display import Image, display
                for path in movie.get("preview_paths") or []:
                    display(Image(filename=path))
            except Exception as exc:
                print("Preview display skipped:", exc)
else:
    F1E_SEQ = None
    F1E_PICK_ID = None
    F1E_PICK_FILENAME = None
    print("F1 probabilities skipped")

### F1e extra — window-to-window fan

Blobs and the coloured XY track now live in the Fig 1e 2 × 2. This cell keeps the existing `plot_focal_fan_switch_networks` helper on the same track (absolute, not a genotype comparison).

In [ ]:
if not (RUN_FIG1_PROBS and RUN_FIG1E_EXTRA):
    print("F1e extra skipped")
elif F1E_SEQ is None or F1E_PICK_ID is None:
    print("F1e extra needs the previous cell (no track selected).")
else:
    seq = F1E_SEQ.copy()
    pick_id = F1E_PICK_ID
    state_col = "transformer_behavior_state"
    if state_col not in seq.columns:
        raise KeyError(f"{state_col!r} missing on the selected track")
    state_seq = seq[state_col].astype(str).tolist()
    order = mixture_state_order(state_seq)
    colors = {s: mixture_state_color(s) for s in order}

    track_scope = "example_track"
    from_s, to_s = state_seq[:-1], state_seq[1:]
    rec = []
    for a in order:
        n_from = sum(1 for x in from_s if x == a)
        for b in order:
            n_ab = sum(1 for x, y in zip(from_s, to_s) if x == a and y == b)
            rec.append({
                "_switch_scope": track_scope,
                "from_state": a,
                "to_state": b,
                "transition_probability": (n_ab / n_from) if n_from else 0.0,
            })
    switch_prob_df = pd.DataFrame(rec)
    comp_df = (
        pd.Series(state_seq, dtype=str)
        .value_counts(normalize=True)
        .rename_axis("state")
        .reset_index(name="fraction")
    )
    comp_df["_switch_scope"] = track_scope
    for s in order:
        if s not in set(comp_df["state"].astype(str)):
            comp_df = pd.concat(
                [comp_df, pd.DataFrame([{"state": s, "fraction": 0.0, "_switch_scope": track_scope}])],
                ignore_index=True,
            )
    switch_df, quantity_col = plasticity.prepare_switch_quantity_table(
        switch_prob_df,
        comp_df,
        group_col="_switch_scope",
        switch_quantity="conditional",
        state_order=order,
    )
    focal = str(pd.Series(state_seq).value_counts().index[0])
    fig_fan, _ = plasticity.plot_focal_fan_switch_networks(
        switch_df,
        comp_df,
        group_col="_switch_scope",
        group_order=[track_scope],
        state_order=order,
        state_colors=colors,
        control_scope=track_scope,
        focal_state=focal,
        display_mode="absolute",
        size_boxes_by_frequency=True,
        show_zero_or_tiny_arrows=True,
        min_arrow_prob=0.01,
        line_color="k",
        probability_col=quantity_col,
        save_path=FIG1_DIR / f"F1e_extra_fan_{VARIANT['feature_mode']}.svg",
        title=f"Fig 1e extra — window-to-window fan   unique_id={pick_id}   focal={focal}",
    )
    plt.show()
    print("Fig 1e extra used unique_id:", pick_id)
    print("Windows in sequence:", len(state_seq), "  distinct assignments:", order)

## Figure 2 — Validation and utility

### F2 confusion matrix versus certain gates

In [ ]:
if RUN_FIG2_CONFUSION:
    label_col = "final_population_certain" if "final_population_certain" in windowed_df.columns else "final_population"
    pred_col = "pred_label" if "pred_label" in windowed_df.columns else "transformer_behavior_state"
    eval_df = windowed_df.dropna(subset=[label_col, pred_col]).copy()
    eval_df = eval_df.loc[eval_df[label_col].isin(PURE_STATES) & eval_df[pred_col].isin(PURE_STATES)]
    print("Evaluation windows (certain ∩ four states):", len(eval_df))
    report = evaluate.plot_confusion_matrix_report(
        eval_df,
        true_col=label_col,
        pred_col=pred_col,
        output_dir=FIG2_DIR,
        name="F2a_heldout_certain_vs_pred",
        labels=PURE_STATES,
        normalize="true",
        figsize=(5.4, 4.8),
        export_format="svg",
    )
    display(report)
    if "is_high_confidence" in eval_df.columns:
        hc = eval_df.loc[eval_df["is_high_confidence"].astype(bool)]
        if len(hc):
            evaluate.plot_confusion_matrix_report(
                hc,
                true_col=label_col,
                pred_col=pred_col,
                output_dir=FIG2_DIR,
                name="F2a_heldout_certain_vs_pred_highconf",
                labels=PURE_STATES,
                normalize="true",
                figsize=(5.4, 4.8),
                export_format="svg",
            )
else:
    print("F2 confusion skipped")

### F2 benchmark

Same split and same held-out windows as the transformer (`python scripts/run_benchmarks.py`).
The random forests on window scalars are upper bounds: the gate labels are drawn on those scalars.

In [ ]:
benchmark_csv = paths["evaluation_dir"] / "benchmarks" / "benchmark_scores.csv"
transformer_scores, _ = evaluate.classification_summary(windowed_df)
rows = [{"method": "transformer_block3_long_large", **transformer_scores}]
if benchmark_csv.exists():
    rows += pd.read_csv(benchmark_csv).to_dict("records")
else:
    print("No benchmark scores yet: python scripts/run_benchmarks.py")
benchmark_df = pd.DataFrame(rows)
benchmark_df.to_csv(FIG2_DIR / "F2b_benchmark_scores.csv", index=False)
display(benchmark_df)

METHOD_LABELS = {
    "transformer_block3_long_large": "Transformer",
    "random_forest_block3_tokens": "RF on\ntokens",
    "gaussian_hmm": "Gaussian\nHMM",
    "random_forest_no_gate_axes": "RF on features\n(no gate axes)",
    "random_forest_all_scalars": "RF on all\nfeatures",
}
# Forests on the window features see the gate axes (or features correlated with them): upper bounds, hatched.
UPPER_BOUNDS = ["random_forest_no_gate_axes", "random_forest_all_scalars"]
plot_df = benchmark_df.set_index("method").reindex([m for m in METHOD_LABELS if m in set(benchmark_df["method"])])

fig, ax = plt.subplots(figsize=(5.0, 3.4))
for x, (method, row) in enumerate(plot_df.iterrows()):
    is_upper = method in UPPER_BOUNDS
    ax.bar(x, row["balanced_accuracy"], color="#0072B2" if method.startswith("transformer") else "#bbbbbb",
           hatch="///" if is_upper else None, edgecolor="white" if is_upper else None)
    ax.text(x, row["balanced_accuracy"] + 0.02, f"{row['balanced_accuracy']:.2f}", ha="center", fontsize=8)
ax.set_xticks(range(len(plot_df)), [METHOD_LABELS[m] for m in plot_df.index], fontsize=8)
ax.set_ylim(0, 1.1)
ax.set_ylabel("balanced accuracy")
ax.set_title("Fig 2b — held-out certain windows\nhatched = upper bound (sees the gate features)", fontsize=9)
fig.tight_layout()
save_fig(fig, FIG2_DIR / "F2b_benchmark.svg")

### F2 uncertainty at behavioural boundaries

In [ ]:
if RUN_FIG2_UNCERTAINTY:
    feat = None
    for cand in ["diffusion_coefficient", "radius_of_gyration", "avg_speed_um_s", "straightness_index"]:
        if cand in windowed_df.columns:
            feat = cand
            break
    plot_df = windowed_df.dropna(subset=["entropy_norm"]).copy()
    if feat:
        plot_df = plot_df.replace([np.inf, -np.inf], np.nan).dropna(subset=[feat])
        plot_df = plot_df.loc[np.isfinite(plot_df[feat])]
        if len(plot_df) > 80_000:
            plot_df = plot_df.sample(80_000, random_state=RANDOM_SEED)
        fig, ax = plt.subplots(figsize=(5.6, 4.2))
        hue = "decision_type" if "decision_type" in plot_df.columns else None
        sns.scatterplot(
            data=plot_df, x=feat, y="entropy_norm", hue=hue,
            palette=DECISION_PALETTE if hue == "decision_type" else None,
            s=6, alpha=0.25, linewidth=0, ax=ax,
        )
        ax.set_title(f"Fig 2c — entropy vs {feat}")
        fig.tight_layout()
        save_fig(fig, FIG2_DIR / f"F2c_entropy_vs_{feat}.svg")

    if {"top1_label", "top2_label", "decision_type"}.issubset(windowed_df.columns):
        pair = windowed_df.assign(
            pair=windowed_df["top1_label"].astype(str) + " / " + windowed_df["top2_label"].astype(str)
        )
        boundary = pair.loc[pair["decision_type"] == "boundary_like_pair"]
        counts = boundary["pair"].value_counts().head(8)
        fig, ax = plt.subplots(figsize=(5.6, 3.6))
        counts.plot(kind="barh", ax=ax, color="#E69F00")
        ax.set_xlabel("n windows")
        ax.set_title("Fig 2c — most common boundary-like pairs")
        fig.tight_layout()
        save_fig(fig, FIG2_DIR / "F2c_boundary_pairs.svg")
        counts.to_csv(FIG2_DIR / "F2c_boundary_pairs.csv")
else:
    print("F2 uncertainty skipped")

### F2 fingerprints — HTT / kinesin / myosin

Uses `mol` (and optional Catherine tables). This is a compositional fingerprint,
not an HD result. If only HTT is in the locked evaluation parquet, the kinesin
and myosin bars stay empty until those prediction tables are pointed at above.

In [ ]:
def fingerprint_table(df, name):
    if "mol" not in df.columns:
        print(name, "has no mol column")
        return pd.DataFrame()
    use = df.copy()
    if "transformer_behavior_state" in use.columns:
        use = use.loc[use["transformer_behavior_state"].isin(PURE_STATES)]
        state_col = "transformer_behavior_state"
    else:
        state_col = "pred_label"
    tab = (
        use.groupby(["mol", state_col]).size().rename("n").reset_index()
    )
    tot = tab.groupby("mol")["n"].transform("sum")
    tab["fraction"] = tab["n"] / tot
    tab["source"] = name
    return tab


if RUN_FIG2_FINGERPRINTS:
    parts = [fingerprint_table(windowed_df, "locked_heldout")]
    if CATHERINE_WINDOWED_PARQUET is not None and Path(CATHERINE_WINDOWED_PARQUET).exists():
        cat = pl.read_parquet(CATHERINE_WINDOWED_PARQUET).to_pandas()
        parts.append(fingerprint_table(cat, "catherine"))
    fp = pd.concat([p for p in parts if len(p)], ignore_index=True)
    if fp.empty:
        print("No fingerprint rows. Check mol column / Catherine path.")
    else:
        fp.to_csv(FIG2_DIR / "F2g_molecule_fingerprints.csv", index=False)
        display(fp)
        state_col = [c for c in fp.columns if c not in ("mol", "n", "fraction", "source")][0]
        for source, sub in fp.groupby("source"):
            pivot = sub.pivot_table(index="mol", columns=state_col, values="fraction", fill_value=0)
            pivot = pivot.reindex(columns=[c for c in PURE_STATES if c in pivot.columns])
            fig, ax = plt.subplots(figsize=(max(3.6, 1.2 * len(pivot)), 3.8))
            pivot.plot(kind="bar", stacked=True, color=[BEHAVIOR_STATE_PALETTE[c] for c in pivot.columns], ax=ax, width=0.8)
            ax.set_ylim(0, 1)
            ax.set_ylabel("fraction of windows")
            ax.set_title(f"Fig 2g — {source}")
            ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", title=None)
            fig.tight_layout()
            save_fig(fig, FIG2_DIR / f"F2g_fingerprints_{source}.svg")
        # Same helper the figuremaker cell uses, if geno is present
        if "geno" in windowed_df.columns:
            stack_df = high_conf_pure(windowed_df)
            if len(stack_df) and stack_df["geno"].nunique() > 1:
                plots.plot_stacked_bar_svg(
                    df=stack_df,
                    x_category="geno",
                    stack_category="transformer_behavior_state",
                    stack_order=BEHAVIOR_STATE_ORDER,
                    colormap=BEHAVIOR_STATE_PALETTE,
                    title=None,
                    export_format="svg",
                    figsize=(5.8, 4),
                    font_size=14,
                    transparent=True,
                    master_dir=FIG2_DIR,
                    percentage_threshold=0.01,
                    line_color="k",
                    custom="F2g_geno_highconf",
                )
else:
    print("F2 fingerprints skipped")

### F2 superwindow / state-switching example

In [ ]:
if RUN_FIG2_SUPERWINDOW:
    sw_source = windowed_df.copy()
    state_col = "transformer_behavior_state"
    if "superwindow_resolved_state" not in sw_source.columns:
        # Carry last pure state over mixture / unresolved labels for a compact example.
        resolved = []
        for uid, grp in sw_source.sort_values(["unique_id", "time_window"]).groupby("unique_id", sort=False):
            last = None
            for st in grp[state_col].astype("string"):
                if str(st) in PURE_STATES:
                    last = str(st)
                    resolved.append(last)
                elif last is not None:
                    resolved.append(last)
                else:
                    resolved.append(pd.NA)
        sw_source = sw_source.sort_values(["unique_id", "time_window"]).copy()
        sw_source["superwindow_resolved_state"] = resolved
        sw_source = sw_source.dropna(subset=["superwindow_resolved_state"])

    sw = superwindows.create_fixed_length_superwindows(
        sw_source,
        superwindow_length=10,
        state_col="superwindow_resolved_state",
        unique_id_col="unique_id",
        time_col="time_window",
        window_uid_col="window_uid",
        additional_cols=present(sw_source, ["mol", "geno", "filename", "type"]),
        min_superwindows=1,
    )
    if isinstance(sw, pl.DataFrame):
        sw = sw.to_pandas()
    print("superwindows:", sw.shape)

    def n_switches(seq):
        seq = list(seq)
        return int(sum(a != b for a, b in zip(seq, seq[1:])))

    sw["n_switches"] = sw["state_sequence"].map(n_switches)
    sw.head(20).to_csv(FIG2_DIR / "F2h_superwindow_sample.csv", index=False)

    # One example sequence
    example = sw.sort_values("n_switches", ascending=False).iloc[0]
    seq = list(example["state_sequence"])
    fig, ax = plt.subplots(figsize=(7.2, 1.6))
    for i, st in enumerate(seq):
        ax.barh(0, 1, left=i, color=BEHAVIOR_STATE_PALETTE.get(st, "#cccccc"), height=0.6)
    ax.set_xlim(0, len(seq))
    ax.set_yticks([])
    ax.set_xlabel("window in superwindow")
    ax.set_title(f"Fig 2h — {example['superwindow_id']}  switches={example['n_switches']}")
    fig.tight_layout()
    save_fig(fig, FIG2_DIR / "F2h_example_sequence.svg")

    if "mol" in sw.columns:
        summary = sw.groupby("mol")["n_switches"].agg(["mean", "median", "count"]).reset_index()
        summary.to_csv(FIG2_DIR / "F2h_switches_by_mol.csv", index=False)
        display(summary)
        fig, ax = plt.subplots(figsize=(4.6, 3.4))
        sns.barplot(data=sw, x="mol", y="n_switches", ax=ax, color="#0072B2")
        ax.set_ylabel("switches / 10-window superwindow")
        ax.set_title("Fig 2h — switching by molecule")
        fig.tight_layout()
        save_fig(fig, FIG2_DIR / "F2h_switches_by_mol.svg")
else:
    print("F2 superwindow skipped")

## Manifest

Re-run this cell after generating panels. Attach the CSV to the paper folder.

In [ ]:
rows = []
for folder in (FIG1_DIR, FIG2_DIR, ED_DIR):
    for p in sorted(folder.glob("*")):
        rows.append({"panel_dir": folder.name, "file": p.name, "suffix": p.suffix, "bytes": p.stat().st_size})
manifest = pd.DataFrame(rows)
manifest.to_csv(RESULTS_DIR / "figure_component_manifest.csv", index=False)
print(manifest.to_string(index=False))